# Flow matching for simple distributions

In [ ]:
import models  # Auxiliary module for neural network architecture and training
import plotting  # Auxiliary module for visualizations

## Prepare toy data

Generate target distribution data (two gaussians) and original noisy data (a standard gaussian)

In [ ]:
import numpy as np

source_data = np.random.randn(1000, 2)

target_data = np.vstack([
    np.random.multivariate_normal([5, 5], np.array([[1, -0.75], [-0.75, 1]]), 500),
    np.random.multivariate_normal([-5, -5], np.array([[1, -0.75], [-0.75, 1]]), 500)
])

plotting.plot_distributions(source_data, target_data)

## Basic flow matching

Let's sample independent couplings between source and target data points

In [ ]:
couplings = models.sample_independent_couplings(source_data, target_data, num_couplings=1000)

Visualize the generated couplings

In [ ]:
plotting.plot_distributions(source_data, target_data, couplings=couplings)

Trains a simple MLP as the velocity field estimation network.

In [ ]:
network = models.train_flow_model(couplings, verbose=True)

Visualize the velocity field we have learned

In [ ]:
plotting.plot_velocity_field(network, source_data, target_data)

Now that we have the velocity field, we can use a integration method such as Euler's method to run across the flow.

In [ ]:
trajectories = models.compute_trajectories(network, source_data)
plotting.plot_trajectories(trajectories)

We can also visualize them as an animation

In [ ]:
plotting.animate_trajectories(trajectories)

We can also see how the synthetic data generated by the flow compares to the original data distribution

In [ ]:
plotting.plot_generated_data_comparison(target_data, trajectories)

## Rectified flows

By using the generated trajectories to learn a new flow, we can improve the "straightness" of the trajectories, allowing for faster generation.

In [ ]:
synthetic_data = np.array([traj[-1][1] for traj in trajectories])
couplings = [(src, tgt) for src, tgt in zip(source_data, synthetic_data)]

rectified_network = models.train_flow_model(couplings, verbose=True)

Let's check now the rectified velocity field

In [ ]:
plotting.plot_velocity_field(rectified_network, source_data, target_data)

The trajectories should be straigther in the new field

In [ ]:
trajectories = models.compute_trajectories(rectified_network, source_data)
plotting.plot_trajectories(trajectories)

In [ ]:
plotting.animate_trajectories(trajectories)

In [ ]:
plotting.plot_generated_data_comparison(target_data, trajectories)